In [1]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import sys
from joblib import parallel_backend
from monte import train_with_cv, fine_tune_with_cv

target cancer

In [2]:
cancer = "BRCA"
trained_metric = "CPE"

training set

In [3]:
df_beta_train = pd.read_parquet("../../data/methylation/train_val_pan-cancer_beta.parquet")
df_meta_train = pd.read_csv("../../data/methylation/train_val_pan-cancer_meta.csv")
df_meta_train = df_meta_train.set_index("Barcode", drop=False)
df_meta_train = df_meta_train.loc[df_beta_train.index]

df_meta_metric = df_meta_train.dropna(subset=[trained_metric])
df_beta_metric = df_beta_train.loc[df_meta_metric["Barcode"]]

In [4]:
# pan-cancer data without the taget cancer
df_beta_pan = df_beta_metric.loc[df_meta_metric["Cancer.type"] != cancer]
df_meta_pan = df_meta_metric.loc[df_meta_metric["Cancer.type"] != cancer]

# target cancer data for bayesian transfer learning
df_beta_cancer = df_beta_metric.loc[df_meta_metric["Cancer.type"] == cancer]
df_meta_cancer = df_meta_metric.loc[df_meta_metric["Cancer.type"] == cancer]

testing set

In [11]:
df_beta_test = pd.read_parquet("../../data/methylation/test_pan-cancer_beta.parquet")
df_meta_test = pd.read_csv("../../data/methylation/test_pan-cancer_meta.csv")
df_meta_test = df_meta_test.set_index("Barcode", drop=False)
df_meta_test = df_meta_test.loc[df_beta_test.index]

In [12]:
df_beta_test_cancer = df_beta_test.loc[df_meta_test["Cancer.type"] == cancer]
df_meta_test_cancer = df_meta_test.loc[df_meta_test["Cancer.type"] == cancer]

pan-cancer model training

In [16]:
pan_model = train_with_cv(df_beta_pan, df_meta_pan[trained_metric])

bayesian transfer on target cancer

In [17]:
cancer_model = fine_tune_with_cv(pan_model, df_beta_cancer, df_meta_cancer[trained_metric])

In [13]:
# probes on sex chromosomes
xy_probes = pd.read_csv('/grain/mk98/cancer-methyl/probe_selection_files/xy_probes.txt', header=None)[0].tolist()

# methmarkerdb
marker_file = "../../data/application/methmarkerdb.literature.230907.tsv"
markers_db = pd.read_csv(marker_file, sep="\t")

# manifest
manifest = pd.read_csv('/grain/mk98/cancer-methyl/probe_selection_files/HM450.hg38.manifest.gencode.v36.tsv.gz', sep='\t', index_col='probeID', compression='gzip')
manifest_genes = manifest[['genesUniq']].copy()
manifest_genes.rename(columns={'genesUniq': 'gene_name'}, inplace=True)

manifest_genes['gene_name'] = manifest_genes['gene_name'].str.split(';').str[0]
manifest_genes = manifest_genes.dropna(subset=['gene_name'])

In [18]:
markers_db["Cancer Type"].value_counts()

Cancer Type
Lung Cancer                                379
Breast Cancer                              350
Colorectal Cancer                          251
Stomach Cancer                             242
Liver Cancer                               143
Prostate Cancer                            130
Esophageal Cancer                          127
Glioblastoma                               111
Cervical Cancer                             94
Urinary Bladder Cancer                      88
Ovarian Cancer                              84
Head and Neck Cancer                        79
Kidney Cancer                               72
Pancreatic Cancer                           60
Acute Myeloid Leukemia                      47
Brain Cancer                                45
Endometrial Cancer                          41
Melanoma                                    29
Thyroid Gland Cancer                        22
Lymphoma                                    18
Squamous Cell Carcinoma                     13
N

In [19]:
def p_to_star(p):
    if p < 1e-4: return "****"
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 0.05: return "*"
    return "ns"

def split_half_dm_corr_perm(meth_mat, labels, B=10, seed=0):
    rng = np.random.RandomState(seed)
    idx_t = labels[labels["group"]==1].index.to_numpy()
    idx_n = labels[labels["group"]==0].index.to_numpy()
    rs = np.empty(B)

    for b in range(B):
        tperm = rng.permutation(idx_t)
        nperm = rng.permutation(idx_n)
        t1,t2 = tperm[:len(tperm)//2], tperm[len(tperm)//2:]
        n1,n2 = nperm[:len(nperm)//2], nperm[len(nperm)//2:]

        def dm(sub_idx):
            lab = labels.loc[sub_idx]
            res = methylize.diff_meth_pos(
                meth_data=meth_mat.loc[sub_idx],
                pheno_data=lab,
                regression_method="linear",
                export=False
            )
            return res["Coefficient"]

        s1 = dm(np.concatenate([t1,n1]))
        s2 = dm(np.concatenate([t2,n2]))
        cc = s1.index.intersection(s2.index)
        rs[b] = spearmanr(s1.loc[cc].values, s2.loc[cc].values).statistic

    return {
        "r_median": float(np.median(rs)),
        "r_q025": float(np.quantile(rs, 0.025)),
        "r_q975": float(np.quantile(rs, 0.975)),
        "r_mean": float(rs.mean()),
        "rs": rs
    }

def pick_closest_promoter_gene(manifest, probes, window=1500):
    m = manifest.loc[probes, ["geneNames","distToTSS"]].dropna()

    out = {}
    for pid, row in m.iterrows():
        genes = str(row["geneNames"]).split(";")
        dists = str(row["distToTSS"]).split(";")
        if len(genes) != len(dists):
            continue
        d = np.array([float(x) for x in dists])
        genes = np.array([g for g in genes])

        # promoter filter
        mask = np.abs(d) <= window
        if not mask.any():
            continue

        # closest TSS
        j = np.argmin(np.abs(d[mask]))
        out[pid] = genes[mask][j]

    return pd.Series(out, name="gene")

In [20]:
# get normal data
normal_path = '/grain/mk98/cancer-methyl/TCGA_Methylation_450K/processed/normal/all'
normal_beta = pd.read_parquet(f"{normal_path}/{cancer}_beta.parquet")
normal_beta = normal_beta.dropna().drop(columns=[x for x in xy_probes if x in normal_beta.columns])
print("normal beta shape:", normal_beta.shape)


test_beta_purified = cancer_model.purify_values(df_beta_test_cancer, alpha=0.05)
print("monte purified shape:", test_beta_purified.shape)

known_markers = markers_db[markers_db["Cancer Type"].str.contains("breast", case=False, na=False, regex=True)]
marker_genes = known_markers["Gene Symbol"].dropna().astype(str).unique().tolist()
print("known markers rows:", known_markers.shape[0], "| n marker genes:", len(marker_genes))

common_probes = df_beta_test_cancer.columns.intersection(test_beta_purified.columns).intersection(normal_beta.columns)
tumor_unadj = df_beta_test_cancer[common_probes].astype(float)
tumor_pur   = test_beta_purified[common_probes].astype(float)
normal      = normal_beta[common_probes].astype(float)
print("Common probes:", len(common_probes))

combined_unadj = pd.concat([tumor_unadj, normal], axis=0)
combined_pur   = pd.concat([tumor_pur,   normal], axis=0)
labels = pd.DataFrame({"group": ([1]*tumor_unadj.shape[0]) + ([0]*normal.shape[0])}, index=combined_unadj.index)

normal beta shape: (97, 293139)
Adjusting 88040 probes passing significance threshold of 0.05.
monte purified shape: (235, 163290)
known markers rows: 350 | n marker genes: 137
Common probes: 163290


In [12]:
combined_unadj.to_parquet(f"../../data/methylation/application/{cancer}_combined_unadj.parquet")
combined_pur.to_parquet(f"../../data/methylation/application/{cancer}_combined_pur.parquet")
labels.to_csv(f"../../data/methylation/application/{cancer}_labels.csv")

Owing to python environmental setting, we seperated the differential methylation analysis code from here and you can check the code in `.ipynb`. The following code would use the output from the differential methylation results.

## Second part

In [5]:
res_unadj = pd.read_csv(f"/grain/wl61/github/ylab/MONTE-analysis/data/methylation/application/{cancer}_diff_meth_unadj.csv", index_col=0)
res_pur = pd.read_csv(f"/grain/wl61/github/ylab/MONTE-analysis/data/methylation/application/{cancer}_diff_meth_pur.csv", index_col=0)

In [21]:
promoter_window=1500
B_perm_marker=5000
B_split=30
seed=0

padj_col = "p_adj" if "p_adj" in res_unadj.columns else ("FDR_QValue" if "FDR_QValue" in res_unadj.columns else None)
assert padj_col is not None, f"Can't find adjusted p-value column: {res_unadj.columns}"

sig_unadj = res_unadj[res_unadj[padj_col] < 0.05]
sig_pur   = res_pur[res_pur[padj_col] < 0.05]
became_sig = sig_pur.index.difference(sig_unadj.index)
lost_sig   = sig_unadj.index.difference(sig_pur.index)

print("DM probes (FDR<0.05) unadj:", sig_unadj.shape[0])
print("DM probes (FDR<0.05) pur  :", sig_pur.shape[0])
print("Became significant after purification:", len(became_sig))
print("Lost significance after purification:", len(lost_sig))

# Marker promoter CpGs (closest promoter gene mapping)
g_u = pick_closest_promoter_gene(manifest, res_unadj.index, window=promoter_window)
g_p = pick_closest_promoter_gene(manifest, res_pur.index,   window=promoter_window)

u_idx = g_u[g_u.isin(marker_genes)].index
p_idx = g_p[g_p.isin(marker_genes)].index
common_marker = u_idx.intersection(p_idx)

eu = np.abs(res_unadj.loc[common_marker, "Coefficient"])
ep = np.abs(res_pur.loc[common_marker, "Coefficient"])
print("Marker promoter CpGs used (paired):", len(common_marker))
print("Median |effect| unadj:", float(np.median(eu)) if len(eu) else np.nan)
print("Median |effect| pur  :", float(np.median(ep)) if len(ep) else np.nan)

# Permutation: do marker CpGs have larger effect size change than random CpGs?
# Universe: promoter-mapped CpGs in both results
g_uni = pick_closest_promoter_gene(manifest, res_unadj.index, window=promoter_window)
uni = g_uni.index.intersection(res_pur.index)
effect_change = np.abs(res_pur.loc[uni, "Coefficient"] - (res_unadj.loc[uni, "Coefficient"]).astype(float))

marker_uni = g_uni.loc[uni][g_uni.loc[uni].isin(marker_genes)].index
n_marker = len(marker_uni)
obs = float(np.median(effect_change.loc[marker_uni])) if n_marker else np.nan

rng = np.random.RandomState(seed)
uni_arr = uni.to_numpy()
null = np.empty(B_perm_marker) if n_marker else np.array([])

sampling_coefficients = {
    "type": ["BRCA markers"] * n_marker,
    "seed": [0] * n_marker,
    "coefficient_change": effect_change.loc[marker_uni].values.tolist()
}

diffs = []
for b in range(B_perm_marker):
    samp = rng.choice(uni_arr, size=n_marker, replace=False)
    null[b] = np.median(effect_change.loc[samp])
    types = ["random"] * n_marker
    seeds = [seed] * n_marker
    coeffs = effect_change.loc[samp].values.tolist()
    diffs.append(np.median(effect_change.loc[samp].values - effect_change.loc[marker_uni].values))
    sampling_coefficients["type"].extend(types)
    sampling_coefficients["seed"].extend(seeds)
    sampling_coefficients["coefficient_change"].extend(coeffs)

p_perm = (1 + np.sum(null >= obs)) / (B_perm_marker + 1)
star = p_to_star(p_perm)

# one random draw for visualization
rand_idx = rng.choice(uni_arr, size=n_marker, replace=False)
marker_vals = effect_change.loc[marker_uni].values
rand_vals   = effect_change.loc[rand_idx].values

DM probes (FDR<0.05) unadj: 119923
DM probes (FDR<0.05) pur  : 129698
Became significant after purification: 16311
Lost significance after purification: 6536
Marker promoter CpGs used (paired): 865
Median |effect| unadj: 0.6335495585384892
Median |effect| pur  : 1.1520788532916444


In [22]:
p_perm

np.float64(0.0001999600079984003)

In [24]:
np.mean(diffs)

np.float64(-0.09982135216555159)

In [22]:
df_sampling_coefficients = pd.DataFrame(sampling_coefficients)

In [25]:
df_summary = pd.DataFrame({
    "BRCA markers": marker_vals,
    "Non-markers": rand_vals
})

In [26]:
df_summary.to_csv(f"../../data/application/{cancer}_marker_effect_change_summary.csv", index=False)
df_sampling_coefficients.to_csv(f"../../data/application/{cancer}_marker_effect_change_sampling.csv", index=False)